**Navigation** : [Sommaire Sudoku](README.md) | [Version Python](Sudoku-18b-Statistical-Comparison-Python.ipynb) | [Sudoku-18](Sudoku-18-Comparison-Python.ipynb)

# Sudoku-18b — Comparaison statistique honnête de solveurs

**Objectifs** : mesurer le bruit et la variabilité entre puzzles ; distinguer un bootstrap de pool d'un bootstrap par puzzle ; comparer des médianes appariées, mesurer un effet et corriger les tests multiples. **Prérequis** : backtracking Sudoku et notions de probabilité. **Durée estimée** : 60 minutes.

Les solveurs, chronométrages et bootstrap sont écrits en C#. SciPy est la référence pour Wilcoxon, Mann-Whitney et les rangs moyens ; statsmodels fournit l'oracle Holm. Python.NET appelle ces bibliothèques et NumPy ne sert qu'à reproduire les masques du Python. Installer Python.NET 3.0.5, NumPy, SciPy et statsmodels dans l'environnement Python ; fournir `PYTHONNET_PYDLL` (bibliothèque partagée Python) et, selon l'installation, `PYTHONHOME` et `PYTHONPATH`. Le noyau initialise le pont une seule fois et libère le verrou pour les appels ultérieurs.

## 1. Mêmes calculs, chronos variables

Un solveur naïf déterministe parcourt les cases puis les chiffres en ordre croissant. On masque 44 positions avec la graine NumPy 1000, comme en Python, puis on répète 30 fois la résolution. Les 37 indices ne garantissent ni unicité ni niveau de difficulté.

In [1]:
#r "nuget: pythonnet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Text;
using Python.Runtime;
using Microsoft.DotNet.Interactive.Formatting;
if (!PythonEngine.IsInitialized) {
 var dll=Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
 if (string.IsNullOrWhiteSpace(dll)) throw new InvalidOperationException("Définir PYTHONNET_PYDLL vers la bibliothèque partagée de Python");
 Runtime.PythonDLL=dll; PythonEngine.Initialize(); PythonEngine.BeginAllowThreads();
}
public static class RefStats {
 static PyList List(double[] a) => new PyList(a.Select(x=>(PyObject)new PyFloat(x)).ToArray());
 public static int[] Mask(int seed) { using(Py.GIL()) { dynamic np=Py.Import("numpy"); dynamic rng=np.random.default_rng(seed); dynamic a=rng.choice(81,size:44,replace:false); return ((PyObject)a.tolist()).As<int[]>(); } }
 public static (double statistic,double p) Wilcoxon(double[] x,double[] y) { using(Py.GIL()) { dynamic st=Py.Import("scipy.stats"); using var a=List(x); using var b=List(y); dynamic r=st.wilcoxon(a,b,zero_method:"wilcox",alternative:"two-sided",method:"auto"); return (((PyObject)r.statistic).As<double>(),((PyObject)r.pvalue).As<double>()); } }
 public static double MannWhitney(double[] x,double[] y) { using(Py.GIL()) { dynamic st=Py.Import("scipy.stats"); using var a=List(x); using var b=List(y); dynamic r=st.mannwhitneyu(a,b,alternative:"two-sided",method:"auto"); return ((PyObject)r.pvalue).As<double>(); } }
 public static double[] Ranks(double[] x) { using(Py.GIL()) { dynamic st=Py.Import("scipy.stats"); using var a=List(x); dynamic r=st.rankdata(a); return ((PyObject)r.tolist()).As<double[]>(); } }
 public static double[] Holm(double[] x) { using(Py.GIL()) { dynamic sm=Py.Import("statsmodels.stats.multitest"); using var a=List(x); dynamic r=sm.multipletests(a,alpha:0.05,method:"holm"); return ((PyObject)r[1].tolist()).As<double[]>(); } }
 public static void Versions() { using(Py.GIL()) { dynamic np=Py.Import("numpy"), sp=Py.Import("scipy"), sm=Py.Import("statsmodels"); Console.WriteLine($".NET {Environment.Version} ; Python.NET {typeof(PythonEngine).Assembly.GetName().Version} ; NumPy {np.__version__} ; SciPy {sp.__version__} ; statsmodels {sm.__version__}"); } }
}
public static class Sudoku {
 static readonly int[] Solution = {5,3,4,6,7,8,9,1,2,6,7,2,1,9,5,3,4,8,1,9,8,3,4,2,5,6,7,8,5,9,7,6,1,4,2,3,4,2,6,8,5,3,7,9,1,7,1,3,9,2,4,8,5,6,9,6,1,5,3,7,2,8,4,2,8,7,4,1,9,6,3,5,3,4,5,2,8,6,1,7,9};
 public static int[] Puzzle(int seed) { var p=(int[])Solution.Clone(); foreach(int i in RefStats.Mask(seed)) p[i]=0; return p; }
 static bool Valid(int[] g,int cell,int n) { int r=cell/9,c=cell%9; for(int i=0;i<9;i++) if(g[r*9+i]==n || g[i*9+c]==n) return false; int br=r/3*3,bc=c/3*3; for(int i=0;i<3;i++) for(int j=0;j<3;j++) if(g[(br+i)*9+bc+j]==n) return false; return true; }
 public static bool Solve(int[] g,int mode) {
  if(mode != 1) {
   for(int cell=0;cell<81;cell++) if(g[cell]==0) {
    for(int k=1;k<=9;k++) {int n=mode==2 ? 10-k : k; if(Valid(g,cell,n)) {g[cell]=n;if(Solve(g,mode))return true;g[cell]=0;}}
    return false;
   }
   return true;
  }
  int best=-1; int[] candidates=null;
  for(int i=0;i<81;i++) if(g[i]==0) {var cand=Enumerable.Range(1,9).Where(n=>Valid(g,i,n)).ToArray();if(candidates==null || cand.Length<candidates.Length){best=i;candidates=cand;}}
  if(best<0)return true;foreach(int n in candidates){g[best]=n;if(Solve(g,mode))return true;g[best]=0;}return false;
 }
 public static bool Check(int[] g,int[] p) { if(g.Length!=81 || Enumerable.Range(0,81).Any(i=>p[i]!=0 && p[i]!=g[i]))return false; var expected=Enumerable.Range(1,9).ToArray(); for(int i=0;i<9;i++) { if(!Enumerable.Range(0,9).Select(j=>g[9*i+j]).OrderBy(x=>x).SequenceEqual(expected))return false; if(!Enumerable.Range(0,9).Select(j=>g[9*j+i]).OrderBy(x=>x).SequenceEqual(expected))return false; } for(int br=0;br<9;br+=3)for(int bc=0;bc<9;bc+=3){var a=new List<int>();for(int r=0;r<3;r++)for(int c=0;c<3;c++)a.Add(g[(br+r)*9+bc+c]);if(!a.OrderBy(x=>x).SequenceEqual(expected))return false;} return true; }
 public static double Time(int[] p,int mode) {var g=(int[])p.Clone();var sw=Stopwatch.StartNew();bool ok=Solve(g,mode);sw.Stop();if(!ok || !Check(g,p))throw new Exception("Solution ou indices invalides");return sw.Elapsed.TotalMilliseconds;}
}
RefStats.Versions();
Console.WriteLine("Solveurs C# ; contrôles des solutions après chronométrage.");

Installed Packages pythonnet, 3.0.5

.NET 10.0.14 ; Python.NET 3.0.5.0 ; NumPy 2.4.4 ; SciPy 1.17.0 ; statsmodels 0.14.6


Solveurs C# ; contrôles des solutions après chronométrage.


Les utilitaires suivants calculent percentiles et graphiques statiques : interpolation linéaire des percentiles, histogrammes et boîtes sur fond blanc. Le bootstrap utilise un générateur C# distinct de NumPy.

In [2]:
public record SvgFigure(string Content);
Formatter.Register<SvgFigure>((v,w)=>w.Write(v.Content),"text/html");
public static class View {
 static string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
 static string Esc(string s)=>System.Security.SecurityElement.Escape(s);
 public static double Q(double[] a,double q) {var s=a.OrderBy(x=>x).ToArray();double k=(s.Length-1)*q;int i=(int)k;return s[i]+(s[Math.Min(i+1,s.Length-1)]-s[i])*(k-i);}
 public static (double mean,double lo,double hi,double[] draws) Boot(double[] a,int n,int seed) {var rng=new Random(seed);var b=new double[n];for(int i=0;i<n;i++){double sum=0;for(int j=0;j<a.Length;j++)sum+=a[rng.Next(a.Length)];b[i]=sum/a.Length;}return(a.Average(),Q(b,.025),Q(b,.975),b);}
 static void Text(StringBuilder s,double x,double y,string t,int size=12) => s.Append($"<text x='{F(x)}' y='{F(y)}' text-anchor='middle' font-family='sans-serif' font-size='{size}'>{Esc(t)}</text>");
 static void Line(StringBuilder s,double x,double y,double x2,double y2,string color="black")=>s.Append($"<line x1='{F(x)}' y1='{F(y)}' x2='{F(x2)}' y2='{F(y2)}' stroke='{color}'/>");
 static StringBuilder Start(int w=900,int h=400)=>new StringBuilder($"<svg xmlns='http://www.w3.org/2000/svg' width='{w}' height='{h}' viewBox='0 0 {w} {h}'><rect width='{w}' height='{h}' fill='white'/>");
 public static SvgFigure Hist(double[] data,int bins,string title,string xlabel) {var s=Start();double left=90,top=55,w=740,h=270,min=data.Min(),max=data.Max();if(max==min)max=min+1;var counts=new int[bins];foreach(double v in data)counts[Math.Min(bins-1,(int)((v-min)/(max-min)*bins))]++;int ymax=Math.Max(1,counts.Max());Text(s,450,25,title,16);Line(s,left,top,left,top+h);Line(s,left,top+h,left+w,top+h);for(int k=0;k<=4;k++){double y=top+h-k*h/4;Line(s,left-5,y,left+w,y,"#dddddd");Text(s,left-35,y+4,F(ymax*k/4.0));double x=left+k*w/4;Text(s,x,top+h+18,F(min+(max-min)*k/4));}for(int i=0;i<bins;i++){double bw=w/bins;double bh=h*counts[i]/ymax;s.Append($"<rect x='{F(left+i*bw)}' y='{F(top+h-bh)}' width='{F(Math.Max(1,bw-1))}' height='{F(bh)}' fill='#77aadd'/>");}double mean=data.Average();double mx=left+(mean-min)/(max-min)*w;Line(s,mx,top,mx,top+h,"#c0392b");Text(s,700,43,"Trait rouge : moyenne (ms)",11);Text(s,450,380,xlabel);Text(s,30,40,"Effectif");s.Append("</svg>");return new SvgFigure(s.ToString());}
 public static SvgFigure Boxes(double[][] values,string[] labels,string title) {var s=Start();double left=90,top=55,w=740,h=270,max=values.SelectMany(x=>x).Max();if(max<=0)max=1;Text(s,450,25,title,16);Line(s,left,top,left,top+h);Line(s,left,top+h,left+w,top+h);for(int k=0;k<=4;k++){double y=top+h-k*h/4;Line(s,left-5,y,left+w,y,"#dddddd");Text(s,left-35,y+4,F(max*k/4));}for(int i=0;i<values.Length;i++){var a=values[i];double x=left+(i+.5)*w/values.Length;double Y(double v)=>top+h-v/max*h;double q1=Q(a,.25),med=Q(a,.5),q3=Q(a,.75),iqr=q3-q1;double low=a.Where(v=>v>=q1-1.5*iqr).Min(),high=a.Where(v=>v<=q3+1.5*iqr).Max();Line(s,x,Y(low),x,Y(high));Line(s,x-8,Y(low),x+8,Y(low));Line(s,x-8,Y(high),x+8,Y(high));s.Append($"<rect x='{F(x-18)}' y='{F(Y(q3))}' width='36' height='{F(Math.Max(.5,Y(q1)-Y(q3)))}' fill='#9ecae1' stroke='black'/>");Line(s,x-18,Y(med),x+18,Y(med));foreach(double v in a.Where(v=>v<low || v>high))s.Append($"<circle cx='{F(x)}' cy='{F(Y(v))}' r='3' fill='white' stroke='black'/>");double ym=Y(a.Average());s.Append($"<polygon points='{F(x)},{F(ym-5)} {F(x+5)},{F(ym)} {F(x)},{F(ym+5)} {F(x-5)},{F(ym)}' fill='#c0392b'/>");Text(s,x,top+h+20,labels[i]);}Text(s,30,40,"ms");Text(s,700,43,"Losange : moyenne ; cercle : valeur hors moustaches",10);Text(s,450,380,"Puzzle / solveur ; moustaches à 1,5 IQR");s.Append("</svg>");return new SvgFigure(s.ToString());}
 public static SvgFigure Bootstrap(double[] data,double[] sizes,double[] widths,double mean) {var s=Start();double left=65,top=60,w=340,h=260,min=data.Min(),max=data.Max();if(max==min)max=min+1;int bins=50;var counts=new int[bins];foreach(double v in data)counts[Math.Min(bins-1,(int)((v-min)/(max-min)*bins))]++;int ymax=Math.Max(1,counts.Max());Text(s,230,25,"Moyennes du pool rééchantillonné",15);Text(s,670,25,"Largeur percentile selon taille",15);Line(s,left,top,left,top+h);Line(s,left,top+h,left+w,top+h);for(int k=0;k<=4;k++){double y=top+h-k*h/4;Text(s,left-25,y+4,F(ymax*k/4.0));Text(s,left+k*w/4,top+h+18,F(min+(max-min)*k/4));}for(int i=0;i<bins;i++){double bw=w/bins,bh=h*counts[i]/ymax;s.Append($"<rect x='{F(left+i*bw)}' y='{F(top+h-bh)}' width='{F(Math.Max(1,bw-1))}' height='{F(bh)}' fill='#74bb8e'/>");}foreach(double v in new[]{Q(data,.025),mean,Q(data,.975)}) {double x=left+(v-min)/(max-min)*w;Line(s,x,top,x,top+h,v==mean?"black":"#c0392b");}Text(s,235,44,"Noir : moyenne ; rouge : percentiles 2,5 / 97,5",10);Text(s,230,375,"Moyenne (ms) ; effectif en ordonnée");left=520;Line(s,left,top,left,top+h);Line(s,left,top+h,left+w,top+h);double maxw=Math.Max(widths.Max(),.000001);for(int k=0;k<=4;k++)Text(s,left-27,top+h-k*h/4+4,F(maxw*k/4));for(int i=0;i<sizes.Length;i++){double x=left+(sizes[i]-sizes[0])/(sizes[sizes.Length-1]-sizes[0])*w,y=top+h-widths[i]/maxw*h;if(i>0){double px=left+(sizes[i-1]-sizes[0])/(sizes[sizes.Length-1]-sizes[0])*w,py=top+h-widths[i-1]/maxw*h;Line(s,px,py,x,y,"#7044aa");}s.Append($"<circle cx='{F(x)}' cy='{F(y)}' r='4' fill='#7044aa'/>");Text(s,x,top+h+18,F(sizes[i]));}Text(s,690,375,"Nombre de chronos ; largeur (ms)");s.Append("</svg>");return new SvgFigure(s.ToString());}
}
Console.WriteLine("Bootstrap percentile et figures SVG chargés.");

Bootstrap percentile et figures SVG chargés.


### Observer les 30 répétitions

L'histogramme compte 15 classes de temps et son trait rouge indique la moyenne. La dispersion provient de la mesure d'un même calcul sur cette machine.

In [3]:
var puzzles=Enumerable.Range(0,8).Select(i=>Sudoku.Puzzle(1000+i)).ToArray();
var timesA=Enumerable.Range(0,30).Select(_=>Sudoku.Time(puzzles[0],0)).ToArray();
double avg=timesA.Average(), sd=Math.Sqrt(timesA.Select(x=>(x-avg)*(x-avg)).Average());
Console.WriteLine($"30 runs : min={timesA.Min():F3} ms max={timesA.Max():F3} ms moyenne={avg:F3} ms écart-type={sd:F3} ms CV={100*sd/avg:F1}% max/min={timesA.Max()/timesA.Min():F2}");
display(View.Hist(timesA,15,"Même puzzle : 30 répétitions","Temps (ms)"));

30 runs : min=0.053 ms max=0.622 ms moyenne=0.079 ms écart-type=0.101 ms CV=128.5% max/min=11.81


Même puzzle : 30 répétitions 0 0.053 7 0.195 14 0.337 21 0.48 28 0.622 Trait rouge : moyenne (ms) Temps (ms) Effectif

Sur les **30 répétitions enregistrées**, le coefficient de variation est d’environ **129 %** et le rapport max/min d’environ **11,8** : quelques chronos élevés élargissent l’histogramme. Ces ratios décrivent ce run sur cette machine et ne constituent pas un seuil général de significativité.

### Exercice 1 — plancher du test exact

Pour `n` différences non nulles sans ex æquo, un test signé bilatéral exact dont tous les signes concordent a pour plus petite p-value `2 / 2^n`. Écrire la fonction et tester plusieurs valeurs de `n` (au moins 1).

In [4]:
double? PMin(int n) { return null; } // TODO étudiant
void VerifierPMin(Func<int,double?> f) {int[] ns={1,2,5,8,10,20};var got=ns.Select(f).ToArray();if(got.All(x=>x==null)){Console.WriteLine("Exercice 1 à compléter");return;}for(int i=0;i<ns.Length;i++)if(got[i]==null || !double.IsFinite(got[i].Value) || Math.Abs(got[i].Value-2/Math.Pow(2,ns[i]))>1e-12)throw new Exception($"Exercice 1 : réponse incorrecte pour n={ns[i]}");Console.WriteLine("Exercice 1 : six valeurs vérifiées");}
VerifierPMin(PMin);

Exercice 1 à compléter


## 2. Huit puzzles de 37 indices

Chaque puzzle est résolu 15 fois. Les boîtes montrent l'étendue, les quartiles et la médiane de chaque série ; elles ne certifient pas l'unicité ni une difficulté identique.

In [5]:
var perPuzzle=puzzles.Select(p=>Enumerable.Range(0,15).Select(_=>Sudoku.Time(p,0)).ToArray()).ToArray();
var medians=perPuzzle.Select(a=>View.Q(a,.5)).ToArray();
for(int i=0;i<8;i++)Console.WriteLine($"P{i+1} : médiane={medians[i]:F3} ms ; moyenne={perPuzzle[i].Average():F3} ms");
var all=perPuzzle.SelectMany(x=>x).ToArray(); Console.WriteLine($"Pool {all.Length} chronos : moyenne={all.Average():F3} ms ; médiane={View.Q(all,.5):F3} ms");
display(View.Boxes(perPuzzle,Enumerable.Range(1,8).Select(i=>$"P{i}").ToArray(),"Temps par puzzle : 15 répétitions"));

P1 : médiane=0.068 ms ; moyenne=0.089 ms


P2 : médiane=0.239 ms ; moyenne=0.267 ms


P3 : médiane=0.239 ms ; moyenne=0.245 ms


P4 : médiane=0.020 ms ; moyenne=0.026 ms


P5 : médiane=0.031 ms ; moyenne=0.036 ms


P6 : médiane=0.067 ms ; moyenne=0.074 ms


P7 : médiane=0.150 ms ; moyenne=0.155 ms


P8 : médiane=0.032 ms ; moyenne=0.037 ms


Pool 120 chronos : moyenne=0.116 ms ; médiane=0.068 ms


Temps par puzzle : 15 répétitions 0 0.126 0.253 0.379 0.505 P1 P2 P3 P4 P5 P6 P7 P8 ms Losange : moyenne ; cercle : valeur hors moustaches Puzzle / solveur ; moustaches à 1,5 IQR

Les **8 médianes** et moyennes en millisecondes résument chacune 15 répétitions. Les boîtes vont du premier au troisième quartile ; les moustaches s’arrêtent aux observations dans 1,5 fois l’écart interquartile, les cercles signalent les valeurs au-delà et le losange rouge indique la moyenne. Dans cette exécution, la médiane de P2 est environ **12 fois** celle de P4 : les mêmes 37 indices ne déterminent donc pas le même temps. Le pool mélange 120 mesures répétées sur huit grilles, pas 120 puzzles indépendants.

## 3. Bootstrap du pool et par puzzle

Le rééchantillonnage naïf des 120 temps fait varier les poids des puzzles malgré les 15 répétitions fixes ; son intervalle percentile décrit le pool observé, sans certifier un IC de population de puzzles. Le second calcul rééchantillonne les huit médianes et vise leur moyenne, un estimand distinct ; sa portée dépend du choix des puzzles. Les largeurs des sous-échantillons ne sont pas garanties monotones.

In [6]:
var boot=View.Boot(all,5000,42); var puzzleBoot=View.Boot(medians,5000,43);
Console.WriteLine($"Pool : moyenne={boot.mean:F3} ms ; percentile 95% [{boot.lo:F3}, {boot.hi:F3}] ms");
Console.WriteLine($"Huit médianes : moyenne={puzzleBoot.mean:F3} ms ; percentile 95% [{puzzleBoot.lo:F3}, {puzzleBoot.hi:F3}] ms");
var sizes=new[]{5,10,20,40,80,120}; var rngSize=new Random(7); var widths=new List<double>();
foreach(int n in sizes){var sub=Enumerable.Range(0,n).Select(_=>all[rngSize.Next(all.Length)]).ToArray();var b=View.Boot(sub,2000,n);widths.Add(b.hi-b.lo);Console.WriteLine($"taille {n} : largeur {b.hi-b.lo:F3} ms");}
display(View.Bootstrap(boot.draws,sizes.Select(x=>(double)x).ToArray(),widths.ToArray(),boot.mean));

Pool : moyenne=0.116 ms ; percentile 95% [0.099, 0.135] ms


Huit médianes : moyenne=0.106 ms ; percentile 95% [0.050, 0.168] ms


taille 5 : largeur 0.100 ms


taille 10 : largeur 0.065 ms


taille 20 : largeur 0.071 ms


taille 40 : largeur 0.057 ms


taille 80 : largeur 0.050 ms


taille 120 : largeur 0.036 ms


Moyennes du pool rééchantillonné Largeur percentile selon taille 0 0.088 73.25 0.104 146.5 0.12 219.75 0.135 293 0.151 Noir : moyenne ; rouge : percentiles 2,5 / 97,5 Moyenne (ms) ; effectif en ordonnée 0 0.025 0.05 0.075 0.1 5 10 20 40 80 120 Nombre de chronos ; largeur (ms)

Un intervalle de confiance fréquentiste désigne une procédure dont la couverture à long terme dépend du modèle et du plan ; après observation on n'attribue pas une probabilité de 95 % au paramètre fixe. Dans la sortie enregistrée, la largeur du percentile du pool représente environ **31 %** de sa moyenne, contre une plage nettement plus large pour les huit médianes ; ce contraste n’établit pas une précision de population. Les deux calculs répondent à des questions différentes et restent conditionnés à la sélection des huit grilles.

## 4. Naïf contre MRV : huit paires

Pour chaque puzzle, chacun des deux solveurs tourne 15 fois : la médiane donne une valeur par solveur et une paire par puzzle. Wilcoxon signé examine les différences appariées ; son interprétation comme test de localisation suppose notamment des différences symétriques sous l'hypothèse nulle. Un non-rejet ne prouve pas l'égalité.

In [7]:
var naive=new double[8];var mrv=new double[8];
for(int i=0;i<8;i++){var a=new double[15];var b=new double[15];for(int j=0;j<15;j++){a[j]=Sudoku.Time(puzzles[i],0);b[j]=Sudoku.Time(puzzles[i],1);}naive[i]=View.Q(a,.5);mrv[i]=View.Q(b,.5);}
var wil=RefStats.Wilcoxon(naive,mrv);var diffs=mrv.Zip(naive,(a,b)=>a-b).Where(x=>x!=0).ToArray();var ranks=RefStats.Ranks(diffs.Select(Math.Abs).ToArray());double plus=0,minus=0;for(int i=0;i<diffs.Length;i++)if(diffs[i]>0)plus+=ranks[i];else minus+=ranks[i];
Console.WriteLine($"n=8 paires ; naïf médiane={View.Q(naive,.5):F3} ms ; MRV médiane={View.Q(mrv,.5):F3} ms ; rapport MRV/naïf={View.Q(mrv,.5)/View.Q(naive,.5):F3}");
Console.WriteLine($"Wilcoxon SciPy W={wil.statistic:F3}, p={wil.p:G5}, {(wil.p<.05?"rejet":"non-rejet")} au seuil 5% ; r signé={(plus+minus>0?(plus-minus)/(plus+minus):double.NaN):F3}");
display(View.Boxes(new[]{naive,mrv},new[]{"Naïf","MRV"},"Médianes par puzzle (ms)"));

n=8 paires ; naïf médiane=0.073 ms ; MRV médiane=0.582 ms ; rapport MRV/naïf=7.931


Wilcoxon SciPy W=0.000, p=0.0078125, rejet au seuil 5% ; r signé=1.000


Médianes par puzzle (ms) 0 0.318 0.635 0.953 1.271 Naïf MRV ms Losange : moyenne ; cercle : valeur hors moustaches Puzzle / solveur ; moustaches à 1,5 IQR

Les deux médianes, en millisecondes, sont agrégées sur **8 puzzles** ; leur rapport MRV/naïf indique le sens observé : supérieur à 1 signifie que MRV a la médiane la plus élevée. Dans le run enregistré, le rapport vaut environ **7,9**, la p-value SciPy vaut **0,0078125** et le rang-bisérial vaut **1** : les huit signes actifs concordent et le test rejette au seuil 0,05. Ce sont deux lectures différentes. Une valeur ±1 indique des signes identiques, pas une amplitude de vitesse universelle.

### Exercice 2 — puissance par simulation

Pour chaque taille 5, 10, 20, 40 et 80, appeler **200 fois** le callback `experience()` et renvoyer la fraction de p-values inférieures à 0,05. Le callback construit deux groupes indépendants lognormaux (`exp(0,3 Z)` avec des normales indépendantes), multiplie le second par l'effet demandé, puis appelle réellement Mann-Whitney. Le harnais contrôle le nombre d'appels et la fraction, pour les effets 1, 1,2 et 1,5 ; le protocole affiché porte sur l'effet 1,2. Le stub ne génère pas lui-même les échantillons.

In [8]:
double? Puissance(int n,double effet,int essais,Func<double> experience) { return null; } // TODO étudiant
void VerifierPuissance(Func<int,double,int,Func<double>,double?> f) {
 int[] ns={5,10,20,40,80}; double[] effets={1.0,1.2,1.5};int absents=0,presents=0;
 foreach(int n in ns) foreach(double effet in effets) {int calls=0,rejets=0;var rng=new Random(10000+n+(int)(effet*100));
 double Normal(){double u1=1-rng.NextDouble(),u2=rng.NextDouble();return Math.Sqrt(-2*Math.Log(u1))*Math.Cos(2*Math.PI*u2);}
 double Experience(){var a=Enumerable.Range(0,n).Select(_=>Math.Exp(.3*Normal())).ToArray();var b=Enumerable.Range(0,n).Select(_=>effet*Math.Exp(.3*Normal())).ToArray();calls++;double p=RefStats.MannWhitney(a,b);if(p<.05)rejets++;return p;}
 double? result=f(n,effet,200,Experience);
 if(result==null){if(calls!=0)throw new Exception("Exercice 2 : résultat absent après appels");absents++;continue;}
 presents++;if(!double.IsFinite(result.Value)||calls!=200||Math.Abs(result.Value-rejets/200.0)>1e-12)throw new Exception($"Exercice 2 : appels ou fraction incorrects pour n={n}, effet={effet}");
 if(effet==1.2)Console.WriteLine($"n={n} : puissance simulée {result:F3}");
 }
 if(presents==0)Console.WriteLine("Exercice 2 à compléter");else if(absents!=0)throw new Exception("Exercice 2 : résultats manquants pour certains cas");
}
VerifierPuissance(Puissance);

Exercice 2 à compléter


## 5. Trois comparaisons et Bonferroni

Le solveur inverse essaie les chiffres en ordre décroissant et fournit lui aussi huit médianes issues de 15 répétitions. Trois tests partagent des données : le risque `1 − 0,95³ ≈ 14 %` vaut pour trois tests indépendants de niveau exact 5 %, et ne calcule pas le risque de cette famille. Bonferroni reste valide sans indépendance des tests.

In [9]:
var reverse=new double[8];for(int i=0;i<8;i++)reverse[i]=View.Q(Enumerable.Range(0,15).Select(_=>Sudoku.Time(puzzles[i],2)).ToArray(),.5);
var comparisons=new[]{("Naïf / MRV",naive,mrv),("Naïf / inverse",naive,reverse),("MRV / inverse",mrv,reverse)};
foreach(var (name,a,b) in comparisons){var p=RefStats.Wilcoxon(a,b).p;double adjusted=Math.Min(1,3*p);Console.WriteLine($"{name,-17} p={p:G5} ; Bonferroni={adjusted:G5} ; {(adjusted<.05?"rejet":"non-rejet")}");}
Console.WriteLine("Le non-rejet ne prouve pas l'égalité des solveurs.");

Naïf / MRV        p=0.0078125 ; Bonferroni=0.023438 ; rejet


Naïf / inverse    p=0.94531 ; Bonferroni=1 ; non-rejet


MRV / inverse     p=0.14844 ; Bonferroni=0.44531 ; non-rejet


Le non-rejet ne prouve pas l'égalité des solveurs.


Les **trois p-values** comparent les paires de solveurs sur les mêmes huit puzzles. La colonne Bonferroni multiplie chacune par 3 et la plafonne à 1 ; le verdict la compare à 0,05. Dans le run enregistré, naïf/MRV donne **0,0234** après correction et les deux comparaisons avec inverse ne rejettent pas. Un non-rejet ne prouve pas une égalité des temps. Une inversion du signe d’une paire peut changer le verdict selon son rang ; elle ne le change pas nécessairement.

### Exercice 3 — p-values ajustées de Holm

Trier les p-values, multiplier successivement par `m-k`, appliquer le maximum cumulatif, plafonner à un puis restaurer l'ordre initial. Renvoyer les valeurs ajustées et les décisions pour `alpha=0,05`. La vérification utilise l'oracle réel `statsmodels` sur des cas non triés, ex æquo, saturation et maximum cumulatif.

In [10]:
(double[] adjusted,bool[] rejected)? HolmExercice(double[] p,double alpha) { return null; } // TODO étudiant
void VerifierHolm(Func<double[],double,(double[] adjusted,bool[] rejected)?> f) {var cases=new[]{new[]{.04,.01,.03},new[]{.01,.01,.8},new[]{.9,.7,.8},new[]{.03,.02,.9,.021},new[]{.001,.9}};var answers=cases.Select(x=>f((double[])x.Clone(),.05)).ToArray();if(answers.All(x=>x==null)){Console.WriteLine("Exercice 3 à compléter");return;}for(int i=0;i<cases.Length;i++){var expected=RefStats.Holm(cases[i]);var r=answers[i];if(r==null||r.Value.adjusted==null||r.Value.rejected==null||r.Value.adjusted.Length!=expected.Length||r.Value.rejected.Length!=expected.Length)throw new Exception($"Exercice 3 : dimensions incorrectes cas {i+1}");for(int j=0;j<expected.Length;j++)if(!double.IsFinite(r.Value.adjusted[j])||Math.Abs(r.Value.adjusted[j]-expected[j])>1e-10||r.Value.rejected[j]!=(expected[j]<.05))throw new Exception($"Exercice 3 : valeur incorrecte cas {i+1}, élément {j+1}");}Console.WriteLine("Exercice 3 : cinq cas Holm vérifiés");}
VerifierHolm(HolmExercice);

Exercice 3 à compléter


## À retenir

- Les répétitions quantifient le bruit sur un puzzle ; huit puzzles donnent huit unités appariées, pas 120.
- Le bootstrap naïf du pool et le bootstrap des médianes par puzzle ont des estimands et limites différents.
- Un test signé requiert ses hypothèses pour une interprétation de localisation ; non-rejet ne signifie pas égalité.
- Bonferroni contrôle la famille sans indépendance des tests ; Holm ajuste aussi les p-values en respectant l'ordre.

## Vérifiez votre compréhension

1. Pourquoi 120 chronos ne sont-ils pas 120 puzzles indépendants ?
2. Quelle différence entre le bootstrap poolé et celui des huit médianes ?
3. Pourquoi une p-value non significative ne démontre-t-elle pas l'égalité ?

<details><summary>Éléments de réponse</summary>

Les répétitions partagent un puzzle ; le pool change les poids des puzzles alors que le second rééchantillonne des unités puzzle. Le non-rejet peut aussi refléter un manque de puissance.
</details>

## Pour aller plus loin

[Sudoku-18 — Comparaison des solveurs](Sudoku-18-Comparison-Python.ipynb) ; Wilcoxon (1945), Mann et Whitney (1947), Efron (1979).